# Mini-TP 6 · Tu modelo en el Data Lake (individual)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

## Consigna
Sube el artefacto de **tu propio modelo** a un Data Lake (MinIO/S3) y **sírvelo cargándolo
desde allí**, no desde disco. Organiza el bucket con **zonas** y **versión**.

### Qué se entrega
1. El bucket con zonas (`raw`, `curated`, `models`) y tu modelo en `models/v1/…`.
2. Una función que **carga el modelo desde el lake** y predice.
3. (Opcional) El experimento registrado con **MLflow apuntando a MinIO/S3**.
4. Una reflexión breve: ¿por qué servir desde el lake y no hornear el modelo en la imagen?

### Se evalúa
- Que **corra de punta a punta**.
- Uso correcto de la API S3 (`boto3`): bucket, zonas por prefijo, versión del modelo.
- Que el modelo se **cargue desde el lake** para servir.
- (Opcional) Integración con MLflow.

> Antes de empezar, levanta MinIO con Docker (ver la *Guía de la práctica* o el tutorial
> `Teoria/datalake_tutorial.ipynb`) y instala las dependencias:
> `uv add boto3 pandas pyarrow scikit-learn mlflow joblib`


## 0. Configuración (dada)
Credenciales por variables de entorno y endpoint de MinIO. No modifiques salvo que tu
MinIO use otro puerto/credenciales.


In [4]:
import os
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ENDPOINT = "http://localhost:9000"
BUCKET = "datalake-tp6"      # usa tu propio bucket
print("Endpoint:", ENDPOINT, "| Bucket:", BUCKET)

Endpoint: http://localhost:9000 | Bucket: datalake-tp6


In [5]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    config=Config(signature_version="s3v4"),
)
print("Cliente S3 creado para", ENDPOINT)


Cliente S3 creado para http://localhost:9000


## 1. Crear el bucket y las zonas


In [6]:
def crear_bucket_y_zonas(bucket):
    existentes = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
    if bucket not in existentes:
        s3.create_bucket(Bucket=bucket)
        print("Bucket creado:", bucket)
    else:
        print("Bucket existente:", bucket)

    for zona in ("raw", "curated", "models"):
        key = f"{zona}/.keep"
        s3.put_object(Bucket=bucket, Key=key, Body=b"")
        print("Zona inicializada:", key)

crear_bucket_y_zonas(BUCKET)


Bucket creado: datalake-tp6
Zona inicializada: raw/.keep
Zona inicializada: curated/.keep
Zona inicializada: models/.keep


## 2. Entrenar (o cargar) tu modelo
Usa **tu modelo** del curso. Si aún no lo tienes a mano, un modelo simple de scikit-learn
sirve para completar el flujo.


In [7]:
from pathlib import Path
import io, json, joblib, numpy as np, pandas as pd


def encontrar_common():
    candidatos = [Path.cwd()/"common", Path.cwd().parent/"common", Path("/mnt/data/common")]
    for p in candidatos:
        if (p/"artifacts"/"model.pkl").exists() and (p/"data"/"healthcare-dataset-stroke-data.csv").exists():
            return p.resolve()
    raise FileNotFoundError("No encontré la carpeta common del proyecto")

COMMON = encontrar_common()
MODEL_LOCAL = COMMON/"artifacts"/"model.pkl"
DATA_DICT_LOCAL = COMMON/"artifacts"/"data_dict.json"
DATASET_LOCAL = COMMON/"data"/"healthcare-dataset-stroke-data.csv"

# Cargamos MI modelo ya entrenado; no se vuelve a entrenar en este TP.
modelo = joblib.load(MODEL_LOCAL)
with open(DATA_DICT_LOCAL, encoding="utf-8") as f:
    data_dict = json.load(f)

df_raw = pd.read_csv(DATASET_LOCAL)
df_raw["bmi"] = df_raw["bmi"].fillna(df_raw["bmi"].median())

# Curated con el mismo preprocesamiento que necesita el modelo.
def preprocesar_dataframe(df_in, metadata):
    feature_fields = ["gender","age","hypertension","heart_disease","ever_married","work_type","Residence_type","avg_glucose_level","bmi","smoking_status"]
    df_new = df_in[feature_fields].copy()
    non_cat = df_new[metadata["non_categorical_columns"]].copy()
    for label, to_drop in metadata["to_drop_map"].items():
        prefix = "is" if len(metadata["to_drop_map"]) > 2 else label
        one_hot = pd.get_dummies(df_new[label], prefix=prefix)
        drop_col = f"{prefix}_{to_drop}"
        if drop_col in one_hot.columns:
            one_hot = one_hot.drop(columns=drop_col)
        df_new = df_new.drop(columns=label).join(one_hot)
    for col in metadata["categorical_columns"]:
        if col not in df_new.columns:
            df_new[col] = 0
    encoded = pd.concat([
        non_cat.reset_index(drop=True),
        df_new[metadata["categorical_columns"]].reset_index(drop=True)
    ], axis=1)
    encoded = encoded[metadata["columns_after_dummy"]].astype(float)
    mean = np.asarray(metadata["standard_scaler_mean"], dtype=float)
    std = np.asarray(metadata["standard_scaler_std"], dtype=float)
    return pd.DataFrame((encoded.to_numpy()-mean)/std, columns=metadata["columns_after_dummy"])

df_curated = preprocesar_dataframe(df_raw, data_dict)
df_curated["stroke"] = df_raw["stroke"].to_numpy()

print("Modelo cargado desde common:", type(modelo).__name__)
print("Nombre/version:", data_dict["model_name"], data_dict["model_version"])
print("Raw:", df_raw.shape, "| Curated:", df_curated.shape)


Modelo cargado desde common: RandomForestClassifier
Nombre/version: stroke_prediction_model_prod 1
Raw: (5110, 12) | Curated: (5110, 17)


## 3. Subir dataset y modelo al lake


In [8]:
# 1) RAW -> CSV en memoria
raw_buffer = io.StringIO()
df_raw.to_csv(raw_buffer, index=False)
RAW_KEY = "raw/stroke/healthcare-dataset-stroke-data.csv"
s3.put_object(Bucket=BUCKET, Key=RAW_KEY, Body=raw_buffer.getvalue().encode("utf-8"), ContentType="text/csv")

# 2) CURATED -> Parquet en memoria
curated_buffer = io.BytesIO()
df_curated.to_parquet(curated_buffer, index=False)
CURATED_KEY = "curated/stroke/stroke_preprocessed.parquet"
s3.put_object(Bucket=BUCKET, Key=CURATED_KEY, Body=curated_buffer.getvalue(), ContentType="application/octet-stream")

# 3) MI MODELO -> models/v1, sin reentrenarlo
with open(MODEL_LOCAL, "rb") as f:
    model_bytes = f.read()
MODEL_KEY = "models/v1/stroke_prediction_model.pkl"
s3.put_object(Bucket=BUCKET, Key=MODEL_KEY, Body=model_bytes, ContentType="application/octet-stream")

# El modelo necesita también el metadata de preprocesamiento.
with open(DATA_DICT_LOCAL, "rb") as f:
    metadata_bytes = f.read()
METADATA_KEY = "models/v1/data_dict.json"
s3.put_object(Bucket=BUCKET, Key=METADATA_KEY, Body=metadata_bytes, ContentType="application/json")

print("Objetos subidos:")
for key in (RAW_KEY, CURATED_KEY, MODEL_KEY, METADATA_KEY):
    print(" -", f"s3://{BUCKET}/{key}")


Objetos subidos:
 - s3://datalake-tp6/raw/stroke/healthcare-dataset-stroke-data.csv
 - s3://datalake-tp6/curated/stroke/stroke_preprocessed.parquet
 - s3://datalake-tp6/models/v1/stroke_prediction_model.pkl
 - s3://datalake-tp6/models/v1/data_dict.json


## 4. Servir el modelo desde el lake
El corazón del Mini-TP: **cargar el modelo desde el lake** (no desde disco) y predecir.


In [9]:
def cargar_modelo_desde_lake(bucket, model_key, metadata_key):
    model_obj = s3.get_object(Bucket=bucket, Key=model_key)
    modelo_lake = joblib.load(io.BytesIO(model_obj["Body"].read()))

    meta_obj = s3.get_object(Bucket=bucket, Key=metadata_key)
    metadata = json.loads(meta_obj["Body"].read().decode("utf-8"))
    return modelo_lake, metadata


modelo_lake, metadata_lake = cargar_modelo_desde_lake(BUCKET, MODEL_KEY, METADATA_KEY)

# Predicción con un registro crudo: el preprocesamiento usa metadata descargada del lake.
ejemplo_raw = df_raw.iloc[[0]].copy()
X_ejemplo = preprocesar_dataframe(ejemplo_raw, metadata_lake)
pred = int(modelo_lake.predict(X_ejemplo)[0])
proba = float(modelo_lake.predict_proba(X_ejemplo)[0, 1])

print("Ejemplo crudo:", ejemplo_raw[["age","avg_glucose_level","bmi"]].to_dict(orient="records")[0])
print("Predicción desde modelo cargado del lake:", pred)
print(f"Probabilidad de stroke: {proba:.4f}")


Ejemplo crudo: {'age': 67.0, 'avg_glucose_level': 228.69, 'bmi': 36.6}
Predicción desde modelo cargado del lake: 0
Probabilidad de stroke: 0.1000


## 5. (Opcional) MLflow apuntando a MinIO/S3


In [10]:
import os, tempfile, mlflow

os.environ["MLFLOW_S3_ENDPOINT_URL"] = ENDPOINT
mlflow.set_tracking_uri("sqlite:///mlflow_tp6.db")

EXPERIMENTO = "mini_tp6_stroke_datalake"
exp = mlflow.get_experiment_by_name(EXPERIMENTO)
if exp is None:
    exp_id = mlflow.create_experiment(EXPERIMENTO, artifact_location=f"s3://{BUCKET}/mlflow")
else:
    exp_id = exp.experiment_id

with mlflow.start_run(experiment_id=exp_id) as run:
    mlflow.log_param("model_name", data_dict["model_name"])
    mlflow.log_param("model_version", data_dict["model_version"])
    mlflow.log_param("model_type", data_dict["model_type"])
    for name, value in data_dict["metrics"].items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(name, float(value))

    # Logueamos exactamente el artefacto existente, no un modelo reentrenado.
    mlflow.log_artifact(str(MODEL_LOCAL), artifact_path="model")
    mlflow.log_artifact(str(DATA_DICT_LOCAL), artifact_path="model")

    print("MLflow run:", run.info.run_id)
    print("Artifact URI:", mlflow.get_artifact_uri())


2026/10/03 15:28:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/03 15:28:40 INFO mlflow.store.db.utils: Updating database tables


MLflow run: 33a9f0c834ec45cbb8ba24173a77b4c2
Artifact URI: s3://datalake-tp6/mlflow/33a9f0c834ec45cbb8ba24173a77b4c2/artifacts


## 6. Reflexión

1. Conviene que el servicio cargue el modelo desde el lake porque separa el ciclo de vida del artefacto del ciclo de vida de la imagen: puedo desplegar una nueva versión del modelo sin reconstruir el contenedor y mantengo una fuente centralizada y auditable de artefactos.
2. Publicaría cada versión en un prefijo inmutable, por ejemplo `models/v1/`, `models/v2/`, y mantendría la versión productiva configurada externamente. Así el rollback consiste en volver a apuntar a una versión anterior sin sobrescribir artefactos.
3. Guardaría las predicciones en `curated/predictions/` (idealmente particionadas por fecha/versión), porque ya son datos derivados y estructurados, listos para monitoreo, auditoría y análisis; `raw` lo reservaría para eventos originales sin transformar.
